# 11 — Geographic Election Outcome Analysis

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("11_geographic_election_outcome_analysis", total_steps=8, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook analyzes the geographic layer of the Senate results: regional/provincial vote strength, turnout, regional candidate rankings, and NCR-vs-national comparison.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
senate = pd.read_csv(RAW / "senate25-final_updated.csv")
candidate_cols = get_candidate_columns(senate)
candidate_ref = build_candidate_reference(senate)
national = pd.read_csv(PROCESSED / "senate_national_candidate_results.csv")
for c in ["registeredVoters", "actualVoters", "validVotes", "overVotes", "underVotes"] + candidate_cols:
    senate[c] = pd.to_numeric(senate[c], errors="coerce")
print(senate.shape)


In [ ]:
progress.step("Purpose")
# Province-level turnout
province_turnout = senate.groupby(["region", "province"], as_index=False).agg(
    registeredVoters=("registeredVoters", "sum"),
    actualVoters=("actualVoters", "sum"),
    validVotes=("validVotes", "sum"),
    overVotes=("overVotes", "sum"),
    underVotes=("underVotes", "sum"),
)
province_turnout["turnout_rate"] = province_turnout["actualVoters"] / province_turnout["registeredVoters"]
province_turnout["undervote_rate"] = province_turnout["underVotes"] / province_turnout["validVotes"]
province_turnout.to_csv(PROCESSED / "province_turnout_metrics.csv", index=False)
province_turnout.sort_values("turnout_rate", ascending=False).head(20)


In [ ]:
progress.step("Purpose")
fig = px.bar(
    province_turnout.sort_values("turnout_rate", ascending=True).tail(30),
    x="turnout_rate", y="province", color="region", orientation="h",
    title="Top 30 provinces by turnout rate",
    labels={"turnout_rate": "Turnout rate", "province": "Province"}
)
fig.update_layout(height=820, xaxis_tickformat=".0%")
save_plotly(fig, INTERACTIVE / "11_top_provinces_by_turnout.html", FIGURES / "11_top_provinces_by_turnout.png")
fig.show()


In [ ]:
progress.step("Purpose")
# Region candidate vote shares for national top 12
region_candidate_totals = senate.groupby("region")[candidate_cols].sum().reset_index()
region_long = region_candidate_totals.melt(id_vars="region", var_name="candidate_column", value_name="votes")
region_long = region_long.merge(candidate_ref[["candidate_column", "candidate", "party"]], on="candidate_column", how="left")
region_totals = region_long.groupby("region", as_index=False)["votes"].sum().rename(columns={"votes": "region_all_candidate_votes"})
region_long = region_long.merge(region_totals, on="region", how="left")
region_long["regional_vote_share"] = region_long["votes"] / region_long["region_all_candidate_votes"]
region_long["regional_rank"] = region_long.groupby("region")["votes"].rank(method="min", ascending=False).astype(int)
region_long.to_csv(PROCESSED / "region_candidate_vote_shares.csv", index=False)
region_long.head()


In [ ]:
progress.step("top12 = national.query('winner_top12 == True')['candidate'].tolist()")
top12 = national.query("winner_top12 == True")["candidate"].tolist()
heat = region_long[region_long["candidate"].isin(top12)].pivot(index="region", columns="candidate", values="regional_vote_share")
plt.figure(figsize=(18, 8))
sns.heatmap(heat, cmap="YlGnBu", annot=False, cbar_kws={"label": "Regional vote share"})
plt.title("Regional vote share heatmap for national top 12 winners")
plt.xlabel("Candidate")
plt.ylabel("Region")
plt.tight_layout()
plt.savefig(FIGURES / "11_regional_vote_share_heatmap_top12.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
progress.step("ncr_mask = region_long['region'].astype(str).str.contains('NCR|National Capital', case=False, na=False)")
# NCR vs national comparison, if NCR exists in region names
ncr_mask = region_long["region"].astype(str).str.contains("NCR|National Capital", case=False, na=False)
if ncr_mask.any():
    ncr = region_long[ncr_mask].copy()
    ncr_rank = ncr.groupby("candidate", as_index=False)["votes"].sum()
    ncr_rank["ncr_rank"] = ncr_rank["votes"].rank(method="min", ascending=False).astype(int)
    ncr_rank = ncr_rank.merge(national[["candidate", "vote_rank", "total_votes", "winner_top12"]], on="candidate", how="left")
    ncr_rank["ncr_minus_national_rank"] = ncr_rank["vote_rank"] - ncr_rank["ncr_rank"]
    ncr_rank.to_csv(TABLES / "11_ncr_vs_national_candidate_ranks.csv", index=False)
    display(ncr_rank.sort_values("ncr_rank").head(20))
    fig = px.scatter(ncr_rank, x="vote_rank", y="ncr_rank", color="winner_top12", hover_name="candidate", title="NCR rank vs national vote rank")
    fig.add_shape(type="line", x0=1, y0=1, x1=66, y1=66, line=dict(dash="dash"))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(autorange="reversed")
    save_plotly(fig, INTERACTIVE / "11_ncr_vs_national_rank_scatter.html", FIGURES / "11_ncr_vs_national_rank_scatter.png")
    fig.show()
else:
    print("No NCR-like region label found. Inspect region names:")
    print(sorted(region_long["region"].dropna().unique())[:50])


In [ ]:
progress.step("region_top1 = region_long.sort_values(['region', 'regional_rank']).groupby('region', as_index=False).first()")
# Top candidate per region
region_top1 = region_long.sort_values(["region", "regional_rank"]).groupby("region", as_index=False).first()
region_top1.to_csv(TABLES / "11_top_candidate_per_region.csv", index=False)
region_top1[["region", "candidate", "party", "votes", "regional_vote_share", "regional_rank"]]


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
